# Zero Point values

SPDX-FileCopyrightText: 2025 German Aerospace Center (DLR)
SPDX-License-Identifier: MIT

This script synthetizes the findings in SpectralResponse.ipynb (PlatoSim distribution) and PLATO-DLR-PL-TN-0113.

Juan Cabrera, Nicholas Jannsen, Denis Griessbach, Marco Montalto.

In [1]:
import numpy as np

In [2]:
# expected fluxes N-CAM, F-CAMb, F-CAMr
# from Tables 6, 7, 8 in PLATO-DLR-PL-TN-0113
# required EOL
expected_fluxes = np.column_stack( 
    [ [ 2.50E+08, 1.58E+05, 6.29E+04, 2.50E+04, 9.97E+03, 3.97E+03, 1.58E+03], 
      [ 1.01E+08, 6.39E+04, 2.54E+04, 1.01E+04, 4.03E+03, 1.61E+03, 6.39E+02], 
      [ 1.15E+08, 7.27E+04, 2.89E+04, 1.15E+04, 4.59E+03, 1.83E+03, 7.27E+02]])

# V magnitudes used as benchmark
v_magnitudes = np.multiply( np.row_stack( np.hstack( ( 0, np.arange( 8, 13.1)))), np.ones( ( 1, 3)))

# P magnitudes used as benchmark
# from Tables 6, 7, 8 in PLATO-DLR-PL-TN-0113
p_magnitudes = np.column_stack( 
    ( np.hstack( ( 0, np.arange( 7.62, 13))), 
      np.hstack( ( 0, np.arange( 7.88, 13))),
      np.hstack( ( 0, np.arange( 7.4,  13))) ) ) 

# absolute spectral response
# from SpectralResponse.ipynb in PlatoSim
absolute_spectral_response = [ 0.458, 0.5126, 0.3265]

# bandwith
# from SpectralResponse.ipynb in PlatoSim
bandwidth = [ 500, 145, 335]

# exposure time
# 1s because the tables in TN113 are photoelectrons/s
exposure_time = [ 1, 1, 1]

# collecting area
# 12 cm aperture
collecting_area = [ 1.131e-2, 1.131e-2, 1.131e-2]

# fluxm0 used in PlatoSim
fluxm0 = expected_fluxes[ 0, :] / absolute_spectral_response / bandwidth / exposure_time / collecting_area

# direct computation of fluxes
fluxes = fluxm0 * absolute_spectral_response * bandwidth * exposure_time * collecting_area * 10.**( - 0.4 * v_magnitudes)

# relative differences are small
( fluxes - expected_fluxes)/fluxes

array([[ 0.00000000e+00,  0.00000000e+00, -1.29575315e-16],
       [-1.65249764e-03, -2.71955444e-03, -1.92813123e-03],
       [-1.63764111e-03, -1.18040911e-03, -4.60628608e-04],
       [ 0.00000000e+00,  0.00000000e+00, -1.58172992e-16],
       [-1.74030889e-03, -2.26755634e-03, -2.57032353e-03],
       [-1.96026303e-03, -5.78348972e-03, -4.04516556e-03],
       [-1.65249764e-03, -2.71955444e-03, -1.92813123e-03]])

In [3]:
# now, if we want to compute from Plato magnitudes (and not from V magnitudes)
# we need to use different zero points (or account for the colour changes)
v_minus_p = v_magnitudes - p_magnitudes
fluxes_p = fluxm0 * absolute_spectral_response * bandwidth * exposure_time * collecting_area * 10.**( - 0.4 * ( p_magnitudes + v_minus_p))

# relative differences are small
( fluxes_p - expected_fluxes)/fluxes_p

array([[ 0.00000000e+00,  0.00000000e+00, -1.29575315e-16],
       [-1.65249764e-03, -2.71955444e-03, -1.92813123e-03],
       [-1.63764111e-03, -1.18040911e-03, -4.60628608e-04],
       [ 0.00000000e+00,  0.00000000e+00, -1.58172992e-16],
       [-1.74030889e-03, -2.26755634e-03, -2.57032353e-03],
       [-1.96026303e-03, -5.78348972e-03, -4.04516556e-03],
       [-1.65249764e-03, -2.71955444e-03, -1.92813123e-03]])

In [4]:
# altenatively, we can define zero point fluxes for P magnitudes
fluxm0_p = fluxm0 * 10.**( - 0.4 * np.median( v_minus_p, axis = 0))
fluxes_p = fluxm0_p * absolute_spectral_response * bandwidth * exposure_time * collecting_area * 10.**( - 0.4 * p_magnitudes)

# relative differences are small
( fluxes_p - expected_fluxes)/fluxes_p

array([[-4.19057522e-01, -1.16863248e-01, -7.37800829e-01],
       [-1.65249764e-03, -2.71955444e-03, -1.92813123e-03],
       [-1.63764111e-03, -1.18040911e-03, -4.60628608e-04],
       [ 7.27595761e-16, -7.20391843e-16, -3.16345983e-16],
       [-1.74030889e-03, -2.26755634e-03, -2.57032353e-03],
       [-1.96026303e-03, -5.78348972e-03, -4.04516556e-03],
       [-1.65249764e-03, -2.71955444e-03, -1.92813123e-03]])

In [5]:
# comparing values
print( 'fluxm0 values to be used when using as input V magnitudes')
print( '[ N-CAM, F-CAM blue, F-CAM red]')
print( fluxm0)

print( '\nfluxm0 values to be used when using as input PLATO magnitudes')
print( '[ N-CAM (PLATO mag), F-CAM blue (PLATO mag blue), F-CAM red (PLATO mag red)]')
print( fluxm0_p)

fluxm0 values to be used when using as input V magnitudes
[ N-CAM, F-CAM blue, F-CAM red]
[9.65254692e+07 1.20146788e+08 9.29623819e+07]

fluxm0 values to be used when using as input PLATO magnitudes
[ N-CAM (PLATO mag), F-CAM blue (PLATO mag blue), F-CAM red (PLATO mag red)]
[6.80208291e+07 1.07575201e+08 5.34942672e+07]
